# Downloading weather from open metero and calculating weather severity index 



In [3]:
"""
Fetch historical weather data from Open-Meteo and compute Weather Severity Index (WSI)
Cities  : Chongqing, Shanghai, Hangzhou
Period  : 21 Mar 2021 – 20 Apr 2021
Output  : One CSV per city + temporal join example
"""

import urllib.request
import json
import csv
import math
import os

# ── Config ────────────────────────────────────────────────────────────────────
CITIES = [
    {"name": "Chongqing", "latitude": 29.5630, "longitude": 106.5516},
    {"name": "Shanghai",  "latitude": 31.2304, "longitude": 121.4737},
    {"name": "Hangzhou",  "latitude": 30.2741, "longitude": 120.1551},
]

START_DATE = "2021-03-17"
END_DATE   = "2021-04-20"

HOURLY_VARS = [
    "temperature_2m",
    "precipitation",
    "rain",
    "weathercode",
    "windspeed_10m",
]

BASE_URL = "https://archive-api.open-meteo.com/v1/archive"

WSI_WEIGHTS = {
    "precipitation": 0.30,
    "windspeed_10m": 0.25,
    "weathercode":   0.25,
    "temperature":   0.20,
}

WEATHERCODE_SEVERITY = {
    0: 0.00, 1: 0.05, 2: 0.10, 3: 0.15,
    45: 0.55, 48: 0.65,
    51: 0.25, 53: 0.30, 55: 0.40,
    56: 0.50, 57: 0.60,
    61: 0.35, 63: 0.45, 65: 0.55,
    66: 0.60, 67: 0.70,
    71: 0.55, 73: 0.65, 75: 0.80,
    77: 0.75,
    80: 0.45, 81: 0.55, 82: 0.70,
    85: 0.70, 86: 0.85,
    95: 0.85, 96: 0.90, 99: 1.00,
}

OUTPUT_FIELDS = [
    "city", "datetime",
    "temperature_2m", "precipitation", "rain", "weathercode", "windspeed_10m",
    "norm_temperature_dev", "norm_precipitation", "norm_windspeed", "norm_weathercode_sev",
    "WSI",
]

# ── Helpers ───────────────────────────────────────────────────────────────────
def weathercode_to_severity(code):
    if code is None:
        return 0.0
    code = int(code)
    if code in WEATHERCODE_SEVERITY:
        return WEATHERCODE_SEVERITY[code]
    nearest = min(WEATHERCODE_SEVERITY.keys(), key=lambda k: abs(k - code))
    return WEATHERCODE_SEVERITY[nearest]


def minmax_normalize(values):
    valid = [v for v in values if v is not None]
    mn, mx = min(valid), max(valid)
    if mx == mn:
        return [0.0 if v is not None else None for v in values]
    return [(v - mn) / (mx - mn) if v is not None else None for v in values]


def build_url(city):
    params = (
        f"latitude={city['latitude']}"
        f"&longitude={city['longitude']}"
        f"&start_date={START_DATE}"
        f"&end_date={END_DATE}"
        f"&hourly={','.join(HOURLY_VARS)}"
        f"&timezone=Asia%2FShanghai"
    )
    return f"{BASE_URL}?{params}"


def fetch_json(url):
    print(f"  Fetching: {url[:90]}...")
    with urllib.request.urlopen(url, timeout=60) as resp:
        return json.loads(resp.read().decode())


def compute_wsi(records):
    """Compute WSI sub-scores and final WSI for a list of records (single city)."""
    temp_dev   = [abs(r["temperature_2m"] - 20) if r["temperature_2m"] is not None else 0 for r in records]
    precip_raw = [r["precipitation"] if r["precipitation"] is not None else 0 for r in records]
    wind_raw   = [r["windspeed_10m"] if r["windspeed_10m"] is not None else 0 for r in records]
    wcode_sev  = [weathercode_to_severity(r["weathercode"]) for r in records]

    norm_temp   = minmax_normalize(temp_dev)
    norm_precip = minmax_normalize(precip_raw)
    norm_wind   = minmax_normalize(wind_raw)
    norm_wcode  = minmax_normalize(wcode_sev)

    w = WSI_WEIGHTS
    for i, r in enumerate(records):
        t  = norm_temp[i]   or 0
        p  = norm_precip[i] or 0
        wi = norm_wind[i]   or 0
        wc = norm_wcode[i]  or 0

        r["norm_temperature_dev"] = round(t,  4)
        r["norm_precipitation"]   = round(p,  4)
        r["norm_windspeed"]       = round(wi, 4)
        r["norm_weathercode_sev"] = round(wc, 4)
        r["WSI"]                  = round(w["temperature"]*t + w["precipitation"]*p +
                                          w["windspeed_10m"]*wi + w["weathercode"]*wc, 4)
    return records


def save_city_csv(city_name, records, out_dir):
    filename = os.path.join(out_dir, f"{city_name.lower()}_wsi_mar17_apr20_2021.csv")
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=OUTPUT_FIELDS)
        writer.writeheader()
        writer.writerows(records)
    print(f"  Saved → {filename}  ({len(records)} rows)")
    return filename


def print_summary(city_name, records):
    wsi_vals = [r["WSI"] for r in records]
    mean = sum(wsi_vals) / len(wsi_vals)
    std  = math.sqrt(sum((v - mean) ** 2 for v in wsi_vals) / len(wsi_vals))
    print(f"  {city_name:12s}  mean WSI={mean:.4f}  std={std:.4f}  "
          f"min={min(wsi_vals):.4f}  max={max(wsi_vals):.4f}")


# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    out_dir = "../dataset/ml/weather-outputs/"

    print("=" * 60)
    print("  Weather Severity Index — per-city CSVs")
    print(f"  Period : {START_DATE} → {END_DATE}")
    print("=" * 60)

    saved_files = []

    for city in CITIES:
        print(f"\n[{city['name']}]")
        data   = fetch_json(build_url(city))
        hourly = data["hourly"]
        n      = len(hourly["time"])

        records = []
        for i in range(n):
            records.append({
                "city":           city["name"],
                "datetime":       hourly["time"][i],
                "temperature_2m": hourly["temperature_2m"][i],
                "precipitation":  hourly["precipitation"][i],
                "rain":           hourly["rain"][i],
                "weathercode":    hourly["weathercode"][i],
                "windspeed_10m":  hourly["windspeed_10m"][i],
            })

        records = compute_wsi(records)
        path    = save_city_csv(city["name"], records, out_dir)
        saved_files.append(path)

    print("\n── WSI Summary ───────────────────────────────────────────────")
    # Re-read for summary (already saved)
    for city in CITIES:
        path = [p for p in saved_files if city["name"].lower() in p][0]
        with open(path, newline="", encoding="utf-8") as f:
            rows = list(csv.DictReader(f))
        print_summary(city["name"], [{"WSI": float(r["WSI"])} for r in rows])

    print(f"\n3 city CSVs saved to: {out_dir}")
    print("Done ✓")


if __name__ == "__main__":
    main()

  Weather Severity Index — per-city CSVs
  Period : 2021-03-17 → 2021-04-20

[Chongqing]
  Fetching: https://archive-api.open-meteo.com/v1/archive?latitude=29.563&longitude=106.5516&start_dat...
  Saved → ../dataset/ml/weather-outputs/chongqing_wsi_mar17_apr20_2021.csv  (840 rows)

[Shanghai]
  Fetching: https://archive-api.open-meteo.com/v1/archive?latitude=31.2304&longitude=121.4737&start_da...
  Saved → ../dataset/ml/weather-outputs/shanghai_wsi_mar17_apr20_2021.csv  (840 rows)

[Hangzhou]
  Fetching: https://archive-api.open-meteo.com/v1/archive?latitude=30.2741&longitude=120.1551&start_da...
  Saved → ../dataset/ml/weather-outputs/hangzhou_wsi_mar17_apr20_2021.csv  (840 rows)

── WSI Summary ───────────────────────────────────────────────
  Chongqing     mean WSI=0.2504  std=0.0979  min=0.0162  max=0.7124
  Shanghai      mean WSI=0.2498  std=0.0805  min=0.0620  max=0.6985
  Hangzhou      mean WSI=0.2533  std=0.1020  min=0.0401  max=0.7149

3 city CSVs saved to: ../dataset/ml/weath

WSI follows the Linear Additive Weighted Index form, which is the dominant paradigm in environmental and risk index construction. 
- The WSI follows the linear additive composite index methodology established by the OECD/JRC framework (Nardo et al., 2008), where normalized sub-indicators are aggregated with domain-informed weights reflecting their empirically established contribution to road accident risk

precipitation (weight: 0.30 — highest)

- Andrey & Yagar (1993) — foundational study showing rain increases accident rates by 70–100% relative to dry conditions. Published in Accident Analysis & Prevention.
- Hermans et al. (2006) — wet road surfaces reduce tire friction coefficient (μ) from ~0.7 (dry) to ~0.4 (wet), a 43% reduction. This is physics, not just correlation.
- Precipitation received the highest weight (0.30) consistent with its dominant effect in the accident-weather literature (Andrey & Yagar, 1993; Theofilatos & Yannis, 2014) and its direct mechanical pathway through reduced road surface friction.

windspeed_10m (weight: 0.25)

Hassan & Barker (1999) — high crosswinds significantly affect vehicle stability, particularly for trucks and high-sided vehicles. Published in Transport Reviews.
Wind speed at 10m (the WMO standard observational height) is weighted at 0.25, reflecting its established role in vehicle lateral stability loss (Hassan & Barker, 1999) and FHWA severity thresholds (Pisano et al., 2008)

he WMO Present Weather Code was mapped to a continuous severity scale following the empirical rank ordering established by Black & Mote (2015), with thresholds anchored to WMO official code definitions (WMO-No.306). This preserves ordinal severity information while enabling continuous integration into the composite index.

WMO-No.306 (Manual on Codes) — the official source for what each code means. You're not inventing the severity order; fog (45/48), thunderstorm (95–99), snow (71–77) are defined as more severe by WMO itself.

- "The WMO Present Weather Code was mapped to a continuous severity scale following the empirical rank ordering established by Black & Mote (2015), with thresholds anchored to WMO official code definitions (WMO-No.306). This preserves ordinal severity information while enabling continuous integration into the composite index.

Temperature is operationalized as absolute deviation from 20°C — the empirically established thermal neutral point for road safety (Bergel-Hayat et al., 2013) — capturing both cold-weather (frost/ice) and heat-stress risks in a single symmetric term. Its lower weight (0.20) reflects its predominantly indirect causal pathway compared to precipitation

we rejected Z-score normalization. Reason for rejection: Z-scores can produce negative values, which breaks the additive WSI interpretation as a 0–1 severity scale. Hence min max 

Runge et al. (2019) Science Advances — the PCMCI paper itself notes that variable selection and preprocessing are critical; correlated inputs (like precipitation + rain + weathercode all encoding "wet weather") violate the causal sufficiency assumption.


-  precipitation, rain, and weathercode are causally redundant representations of the same upstream physical process (atmospheric moisture → precipitation event). Including all three as separate nodes in PCMCI+ would create spurious edges among them, obscuring the true causal path to accident outcomes. WSI collapses this into a single informative node while temperature_2m and windspeed_10m are retained separately because they represent independent physical mechanisms.


### Limitations 

Limitation: Weights are not data-derived"Weights were set a priori based on domain literature; future work could use PCA loadings or Bayesian weight estimation to make them data-driven"Min-max sensitive to outliers"Robustness was verified using Z-score normalization; Spearman correlation between both WSI variants was ρ=X.XX"WSI loses mechanistic detail"Temperature and windspeed are retained as separate PCMCI+ nodes alongside WSI to preserve mechanistic interpretability"Hourly resolution may miss sub-hourly events"Open-Meteo archive data is hourly; sub-hourly weather variability is a known limitation of ERA5-based reanalysis products"